# P02a Notebook 1: 数据获取

**任务**：获取10只A股股票（≥5行业）5年日度OHLCV数据、沪深300及附加指数、≥2个宏观指标（含CPI）、5年财务指标。

In [1]:
import os
import time
import datetime
import pandas as pd
import akshare as ak
import baostock as bs

# ── 目录结构（自动创建）──────────────────────────────────────────
DIRS = [
    '../data/raw/stocks',
    '../data/raw/index',
    '../data/raw/macro',
    '../data/raw/financial',
    '../data/processed',
    '../data/interim',
    '../reports/figures',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print('目录结构已创建')

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/lee/Library/Python/3.9/lib/python/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/lee/Library/Python/3.9/lib/python/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/lee/Library/Python/3.9/lib/python/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/lee/Library/Python/3.9/lib/python/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/Users/lee/Library/Python/3.9/lib/python/site-packages/traitlets/traitlets.py", line 668, in get
    value = self._validate(obj, default)
  File "/Users/lee/Library

目录结构已创建


In [2]:
# ── 下载日志 ──────────────────────────────────────────────────
log_records = []

def log(source, status, note=''):
    record = {
        'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'source': source,
        'status': status,
        'note': note,
    }
    log_records.append(record)
    print(f"[{record['timestamp']}] {status:8s} | {source} | {note}")

In [3]:
# ── 股票列表（10只，覆盖≥5行业）─────────────────────────────────
# 代码使用akshare A股格式（6位数字字符串）
STOCKS = {
    '600036': '招商银行',      # 银行
    '601398': '工商银行',      # 银行
    '600519': '贵州茅台',      # 白酒
    '000858': '五粮液',        # 白酒
    '600276': '恒瑞医药',      # 医药
    '000625': '长安汽车',      # 汽车
    '600887': '伊利股份',      # 食品
    '601088': '中国神华',      # 能源
    '600028': '中国石化',      # 能源
    '600941': '中国移动',      # 电信
}

END_DATE   = '20241231'
START_DATE = '20200101'  # 5年

print('股票列表：')
for code, name in STOCKS.items():
    print(f'  {code}  {name}')

股票列表：
  600036  招商银行
  601398  工商银行
  600519  贵州茅台
  000858  五粮液
  600276  恒瑞医药
  000625  长安汽车
  600887  伊利股份
  601088  中国神华
  600028  中国石化
  600941  中国移动


In [4]:
# ── Part 1: 用 baostock 下载股票日度OHLCV ────────────────────────
# baostock 代码格式: sh.600036 / sz.000858

def bs_code(code):
    """将6位代码转为 baostock 格式"""
    if code.startswith('6'):
        return f'sh.{code}'
    else:
        return f'sz.{code}'

# 登录 baostock
lg = bs.login()
print(f'baostock 登录: {lg.error_code} {lg.error_msg}')

for code, name in STOCKS.items():
    save_path = f'../data/raw/stocks/{code}.csv'
    if os.path.exists(save_path):
        log(f'stock/{code}', 'SKIP', '文件已存在')
        continue
    try:
        rs = bs.query_history_k_data_plus(
            bs_code(code),
            "date,open,high,low,close,volume,amount",
            start_date=START_DATE[:4]+'-'+START_DATE[4:6]+'-'+START_DATE[6:],
            end_date=END_DATE[:4]+'-'+END_DATE[4:6]+'-'+END_DATE[6:],
            frequency="d",
            adjustflag="2",   # 前复权
        )
        rows = []
        while (rs.error_code == '0') and rs.next():
            rows.append(rs.get_row_data())
        df = pd.DataFrame(rows, columns=rs.fields)
        if df.empty:
            log(f'stock/{code}', 'FAILED', '返回空数据')
        else:
            df.to_csv(save_path, index=False, encoding='utf-8-sig')
            log(f'stock/{code}', 'SUCCESS', f'{name} 共{len(df)}行')
    except Exception as e:
        log(f'stock/{code}', 'FAILED', str(e))

bs.logout()
print('baostock 已登出')

login success!
baostock 登录: 0 success
[2026-05-17 19:53:35] SKIP     | stock/600036 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/601398 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/600519 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/000858 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/600276 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/000625 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/600887 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/601088 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/600028 | 文件已存在
[2026-05-17 19:53:35] SKIP     | stock/600941 | 文件已存在
logout success!
baostock 已登出


In [5]:
# ── Part 2: 用 baostock 下载指数 ──────────────────────────────
INDICES = {
    'sh.000300': '沪深300',
    'sh.000001': '上证指数',
}

lg2 = bs.login()
for bs_sym, name in INDICES.items():
    code = bs_sym.split('.')[1]
    save_path = f'../data/raw/index/{code}.csv'
    if os.path.exists(save_path):
        log(f'index/{code}', 'SKIP', '文件已存在')
        continue
    try:
        rs = bs.query_history_k_data_plus(
            bs_sym,
            "date,open,high,low,close,volume",
            start_date=START_DATE[:4]+'-'+START_DATE[4:6]+'-'+START_DATE[6:],
            end_date=END_DATE[:4]+'-'+END_DATE[4:6]+'-'+END_DATE[6:],
            frequency="d",
        )
        rows = []
        while (rs.error_code == '0') and rs.next():
            rows.append(rs.get_row_data())
        df = pd.DataFrame(rows, columns=rs.fields)
        if df.empty:
            log(f'index/{code}', 'FAILED', '返回空数据')
        else:
            df.to_csv(save_path, index=False, encoding='utf-8-sig')
            log(f'index/{code}', 'SUCCESS', f'{name} 共{len(df)}行')
    except Exception as e:
        log(f'index/{code}', 'FAILED', str(e))

bs.logout()
print('指数下载完成')

login success!
[2026-05-17 19:53:36] SKIP     | index/000300 | 文件已存在
[2026-05-17 19:53:36] SKIP     | index/000001 | 文件已存在
logout success!
指数下载完成


In [6]:
# ── Part 3: 宏观指标 ──────────────────────────────────────────
# 3a: CPI（必选）
try:
    cpi = ak.macro_china_cpi_monthly()
    cpi.to_csv('../data/raw/macro/cpi_monthly.csv', index=False, encoding='utf-8-sig')
    log('macro/CPI', 'SUCCESS', f'共{len(cpi)}行')
except Exception as e:
    log('macro/CPI', 'FAILED', str(e))

time.sleep(0.3)

# 3b: M2货币供应量（自选）
try:
    m2 = ak.macro_china_money_supply()
    m2.to_csv('../data/raw/macro/m2_monthly.csv', index=False, encoding='utf-8-sig')
    log('macro/M2', 'SUCCESS', f'共{len(m2)}行')
except Exception as e:
    log('macro/M2', 'FAILED', str(e))

[2026-05-17 19:53:38] SUCCESS  | macro/CPI | 共357行
[2026-05-17 19:53:39] SUCCESS  | macro/M2 | 共220行


In [7]:
# ── Part 4: 财务指标（近5年，长格式）─────────────────────────────
# 要求: ≥2个财务指标，以长格式存储（股票代码、年度、指标名、值）

fin_records = []
lg3 = bs.login()
for code, name in STOCKS.items():
    try:
        # 用 baostock 获取年度财务数据（ROE、ROA等）
        rs = bs.query_profit_data(
            code=bs_code(code),
            year=2024,
            quarter=4,
        )
        rows = []
        while (rs.error_code == '0') and rs.next():
            rows.append(rs.get_row_data())
        if not rows:
            raise ValueError('空数据')
        df_fin = pd.DataFrame(rows, columns=rs.fields)
        df_fin['股票代码'] = code
        df_fin['股票名称'] = name
        fin_records.append(df_fin)
        log(f'financial/{code}', 'SUCCESS', f'{name} 共{len(df_fin)}行')
    except Exception as e:
        log(f'financial/{code}', 'FAILED', str(e))
bs.logout()

# 5年数据：2020-2024，逐年获取
fin_all_years = []
for year in range(2020, 2025):
    lg4 = bs.login()
    for code, name in STOCKS.items():
        try:
            rs = bs.query_profit_data(code=bs_code(code), year=year, quarter=4)
            rows = []
            while (rs.error_code == '0') and rs.next():
                rows.append(rs.get_row_data())
            if rows:
                df_y = pd.DataFrame(rows, columns=rs.fields)
                df_y['股票代码'] = code
                df_y['股票名称'] = name
                df_y['year'] = year
                fin_all_years.append(df_y)
        except Exception:
            pass
    bs.logout()

if fin_all_years:
    fin_wide = pd.concat(fin_all_years, ignore_index=True)
    # 转为长格式：指标名作一列
    id_cols = ['股票代码', '股票名称', 'year', 'statDate']
    id_cols = [c for c in id_cols if c in fin_wide.columns]
    val_cols = [c for c in fin_wide.columns if c not in id_cols]
    fin_long = fin_wide.melt(
        id_vars=id_cols,
        value_vars=val_cols,
        var_name='indicator',
        value_name='value',
    )
    fin_long.to_csv('../data/raw/financial/financial_indicators.csv',
                    index=False, encoding='utf-8-sig')
    log('financial/all', 'SUCCESS', f'长格式财务数据 {fin_long.shape}')
    print(fin_long.head())
else:
    log('financial/all', 'FAILED', '所有股票财务数据获取失败')
    print('财务数据获取失败，跳过')

login success!
[2026-05-17 19:53:39] SUCCESS  | financial/600036 | 招商银行 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/601398 | 工商银行 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/600519 | 贵州茅台 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/000858 | 五粮液 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/600276 | 恒瑞医药 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/000625 | 长安汽车 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/600887 | 伊利股份 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/601088 | 中国神华 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/600028 | 中国石化 共1行
[2026-05-17 19:53:39] SUCCESS  | financial/600941 | 中国移动 共1行
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
login success!
logout success!
[2026-05-17 19:53:43] SUCCESS  | financial/all | 长格式财务数据 (490, 6)
     股票代码  股票名称  year    statDate indicator      value
0  600036  招商银行  2020  2020-12-31      code  sh.600036
1  601398  工商银行  2020  2020-1

In [8]:
# ── 保存下载日志 ──────────────────────────────────────────────
log_df = pd.DataFrame(log_records)
log_df.to_csv('../data/raw/download_log.csv', index=False, encoding='utf-8-sig')
print('\n===== 下载汇总 =====')
print(log_df['status'].value_counts())
print('\n下载日志已保存至 data/raw/download_log.csv')
log_df


===== 下载汇总 =====
status
SUCCESS    13
SKIP       12
Name: count, dtype: int64

下载日志已保存至 data/raw/download_log.csv


,timestamp,source,status,note
0,2026-05-17 19:53:35,stock/600036,SKIP,文件已存在
1,2026-05-17 19:53:35,stock/601398,SKIP,文件已存在
2,2026-05-17 19:53:35,stock/600519,SKIP,文件已存在
3,2026-05-17 19:53:35,stock/000858,SKIP,文件已存在
4,2026-05-17 19:53:35,stock/600276,SKIP,文件已存在
5,2026-05-17 19:53:35,stock/000625,SKIP,文件已存在
6,2026-05-17 19:53:35,stock/600887,SKIP,文件已存在
7,2026-05-17 19:53:35,stock/601088,SKIP,文件已存在
8,2026-05-17 19:53:35,stock/600028,SKIP,文件已存在
9,2026-05-17 19:53:35,stock/600941,SKIP,文件已存在
